# C2 — Task-Conditioned Chunking Inference

Evaluate whether assigning each question to its *optimal* chunking strategy (learned from a training split) improves retrieval over the default baseline.

**Pipeline:**
1. Load questions + stored embeddings from ChromaDB
2. 70/30 train/test split (seed=42)
3. Cluster training questions: UMAP(10D) → HDBSCAN
4. For each cluster, find the chunking strategy with the best Coverage@3 on training questions
5. For each test question, find nearest training cluster → assign its optimal strategy
6. Evaluate 4 conditions on the test set:
   - **A** Baseline + no rerouting
   - **B** Baseline + query rerouting
   - **C** Task-conditioned + no rerouting
   - **D** Task-conditioned + query rerouting

All retrieval is pre-cached in `data/RETRIEVER_results_chunking_comparison/` — **zero API calls**.

## 1. Imports & Configuration

In [1]:
import json
import os
import sys

import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap
from sklearn.metrics.pairwise import euclidean_distances

if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from scripts.config import COHERE_API_KEY, EMBEDDINGS_1887_COHERE
from scripts.utils.chroma_utils import load_chroma_collection
from scripts.retrieval_utils import (
    apply_no_rerouting,
    apply_query_rerouting,
    apply_forced_multicollection,
    apply_forced_equal_from_all_collections,
    compute_recall_metrics_dataframe,
)
from scripts.evaluation_utils import compute_coverage_row_list

print("Imports OK")


c:\Users\plathoon\AppData\Local\anaconda3\envs\debattre\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [10]:
SEED = 42
TRAIN_RATIO = 0.75
K_VALUES = [3, 5, 10]
OPT_METRIC = "coverage@3"  # metric used to pick best strategy per cluster

BASE_DIR = os.getcwd() if os.path.basename(os.getcwd()) != "notebooks" else os.path.join(os.getcwd(), "..")
DATA_DIR = os.path.join(BASE_DIR, "data")
CACHE_DIR = os.path.join(DATA_DIR, "RETRIEVER_results_chunking_comparison")

# All 14 cached strategies.  10000_hierarchical = production baseline.
STRATEGY_IDS = [
    "10000_hierarchical",
    "baseline_10k",
    "S2_atomic", "S3_window3", "S4_president", "S8_vote",
    "fixed_500", "fixed_1000", "fixed_2000",
    "S9_topic", "S9_topic_noVotes",
    "S10_amendment",
    "S11_hybrid", "S11_hybrid_noVotes",
]
BASELINE = "10000_hierarchical"

CLASS_TO_COLLECTIONS = {
    "Le Gaulois + L'Intransigeant": ["Le Gaulois", "L'Intransigeant"],
    "L'Intransigeant + Les Débats": ["Le Gaulois", "L'Intransigeant", "Les Débats"],
    "Le Gaulois + Les Débats":      ["Le Gaulois", "L'Intransigeant", "Les Débats"],
}

# Routing configurations: each entry defines fn, kwargs, and output column names.
# Each TC condition trains AND evaluates under the SAME routing — avoiding mismatch.
ROUTING_CONFIGS = {
    "no_rerouting": {
        "fn": apply_no_rerouting,
        "kwargs": {},
        "docs_col": "documents",
        "sources_col": "sources",
        "label": "No Rerouting",
    },
    "qre": {
        "fn": apply_query_rerouting,
        "kwargs": {"class_to_collections": CLASS_TO_COLLECTIONS},
        "docs_col": "documents_rerouted",
        "sources_col": "sources_rerouted",
        "label": "Query Rerouting (QRe)",
    },
    "ums": {
        "fn": apply_forced_equal_from_all_collections,
        "kwargs": {},
        "docs_col": "documents_forced_equal",
        "sources_col": "sources_forced_equal",
        "label": "UMS (Forced Equal)",
    },
    "qre_ums": {
        "fn": apply_forced_multicollection,
        "kwargs": {"class_to_collections": CLASS_TO_COLLECTIONS},
        "docs_col": "documents_forced",
        "sources_col": "sources_forced",
        "label": "QRe + UMS",
    },
}

print(f"Cache dir: {CACHE_DIR}")
print(f"Strategies: {len(STRATEGY_IDS)}")
print(f"Routing configs: {list(ROUTING_CONFIGS.keys())}")


Cache dir: c:\Users\plathoon\Desktop\Aurelien\debattre-generate-questions\data\RETRIEVER_results_chunking_comparison
Strategies: 14
Routing configs: ['no_rerouting', 'qre', 'ums', 'qre_ums']


## 2. Load Questions + Embeddings from ChromaDB

In [11]:
collection_questions = load_chroma_collection(
    EMBEDDINGS_1887_COHERE, api_key=COHERE_API_KEY, collection_name="questions_MH_v1"
)
result = collection_questions.get(include=["metadatas", "documents", "embeddings"])

questions_df = pd.DataFrame({
    "question":   result["documents"],
    "embeddings": list(result["embeddings"]),
    "metadata":   result["metadatas"],
})
questions_df = pd.concat(
    [questions_df.drop("metadata", axis=1), questions_df["metadata"].apply(pd.Series)],
    axis=1,
)
questions_df = questions_df.loc[:, ~questions_df.columns.duplicated()]
questions_df = questions_df[questions_df["question"] != "AUCUNE QUESTION"].reset_index(drop=True)

if "gold_ids" in questions_df.columns and isinstance(questions_df["gold_ids"].iloc[0], str):
    questions_df["gold_ids"] = questions_df["gold_ids"].apply(json.loads)

print(f"Loaded {len(questions_df)} questions")
print(f"Columns: {questions_df.columns.tolist()}")

Loaded 875 questions
Columns: ['question', 'embeddings', 'unique_id', 'id_2', 'id_1', 'source_1', 'gold_ids', 'question_type', 'question_index', 'source_2']


## 3. Train / Test Split

In [12]:
rng = np.random.default_rng(SEED)
shuffled_idx = rng.permutation(len(questions_df))
n_train = int(len(questions_df) * TRAIN_RATIO)

train_ids = set(questions_df.iloc[shuffled_idx[:n_train]]["unique_id"].tolist())
test_ids  = set(questions_df.iloc[shuffled_idx[n_train:]]["unique_id"].tolist())

train_df = questions_df[questions_df["unique_id"].isin(train_ids)].reset_index(drop=True)
test_df  = questions_df[questions_df["unique_id"].isin(test_ids)].reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")

Train: 656  |  Test: 219


## 4. Cluster Training Questions (UMAP → HDBSCAN)

In [13]:
X_train = np.array(train_df["embeddings"].tolist())

# Reduce to 10D for clustering (fit on train only)
CLUSTER_DIM = 10
reducer = umap.UMAP(n_components=CLUSTER_DIM, metric="cosine", random_state=SEED)
X_train_umap = reducer.fit_transform(X_train)

# HDBSCAN on the reduced space
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=5, metric="euclidean")
train_labels = clusterer.fit_predict(X_train_umap)

n_clusters = len(set(train_labels) - {-1})
n_noise    = (train_labels == -1).sum()
print(f"HDBSCAN → {n_clusters} clusters, {n_noise} noise points")

train_df = train_df.copy()
train_df["cluster"] = train_labels

# Cluster centroids (mean UMAP coords) — used to assign test questions.
# Noise cluster (-1) gets its own centroid so test questions can be routed to it.
centroids = {}
for cid in set(train_labels):
    mask = train_labels == cid
    centroids[cid] = X_train_umap[mask].mean(axis=0)

print(f"Centroids computed for {len(centroids)} clusters (including noise cluster -1)")


c:\Users\plathoon\AppData\Local\anaconda3\envs\debattre\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


HDBSCAN → 20 clusters, 123 noise points
Centroids computed for 21 clusters (including noise cluster -1)


## 5. Find Optimal Strategy per Cluster (on Training Set)

For each cluster — including the HDBSCAN noise cluster (−1) — load each strategy's cached JSONL, filter to training questions, compute Coverage@3, and record the winner. All clusters compete equally; there is no automatic fallback to the baseline.


In [14]:
# Load the original-document collections needed for gold text lookup (Coverage metric)
GOLD_COLLECTION_CONFIGS = [
    {"name": "legaulois_1887_v1",                       "label": "Le Gaulois"},
    {"name": "lintransigeant_1887_v1",                  "label": "L'Intransigeant"},
    {"name": "debattre_1887_10000_hierarchical_cohere", "label": "Les Débats"},
]

gold_collections = {}
for cfg in GOLD_COLLECTION_CONFIGS:
    col = load_chroma_collection(
        EMBEDDINGS_1887_COHERE, api_key=COHERE_API_KEY, collection_name=cfg["name"]
    )
    gold_collections[cfg["label"]] = col
    print(f"  Loaded {cfg['label']} ({col.count()} docs)")
print("Gold lookup collections ready.")


  Loaded Le Gaulois (78 docs)
  Loaded L'Intransigeant (79 docs)
  Loaded Les Débats (5103 docs)
Gold lookup collections ready.


In [15]:
def load_jsonl_as_df(strategy_id):
    """Load cached universal retrieval JSONL for a given strategy."""
    path = os.path.join(CACHE_DIR, f"universal_{strategy_id}.jsonl")
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    df = pd.DataFrame(rows)
    for col in ["gold_ids", "retrieved_ids", "distances", "documents", "sources"]:
        if col in df.columns and isinstance(df[col].iloc[0], str):
            df[col] = df[col].apply(json.loads)
    return df


# Group training question IDs by cluster — includes noise cluster (-1)
cluster_train_qids = {}
for cid in set(train_labels):
    cluster_train_qids[cid] = set(train_df[train_df["cluster"] == cid]["unique_id"].tolist())

# cluster_scores_by_routing[routing_key][cid][sid] = Coverage@3
# Each routing function is applied ONCE per strategy, then per-cluster subsets are scored.
SCORE_K = 3
cluster_scores_by_routing = {
    rk: {cid: {} for cid in cluster_train_qids}
    for rk in ROUTING_CONFIGS
}

print("Computing Coverage@3 per strategy × per cluster × per routing...")
for sid in STRATEGY_IDS:
    strat_df = load_jsonl_as_df(sid)
    strat_uids = set(strat_df["unique_id"].tolist())

    for rk, rcfg in ROUTING_CONFIGS.items():
        # Apply routing ONCE on the full strategy df, then slice per cluster
        routed_full = rcfg["fn"](strat_df.copy(), n_results=SCORE_K, **rcfg["kwargs"])

        for cid, qids in cluster_train_qids.items():
            valid_qids = qids & strat_uids
            subset = routed_full[routed_full["unique_id"].isin(valid_qids)]
            if len(subset) == 0:
                cluster_scores_by_routing[rk][cid][sid] = float("nan")
                continue
            cov = compute_coverage_row_list(
                subset, k=SCORE_K, collections_dict=gold_collections,
                docs_col=rcfg["docs_col"], sources_col=rcfg["sources_col"],
            )
            cluster_scores_by_routing[rk][cid][sid] = float(cov[2].mean())

    print(f"  {sid} done")

# Backward-compat alias (used by UMAP cell and summary table)
cluster_scores = cluster_scores_by_routing["no_rerouting"]
print("Done.")


Computing Coverage@3 per strategy × per cluster × per routing...


  10000_hierarchical done


  baseline_10k done


  S2_atomic done


  S3_window3 done


...

  S9_topic done


  S9_topic_noVotes done


  S10_amendment done


  S11_hybrid done


  S11_hybrid_noVotes done
Done.


In [18]:
cluster_scores

{0: {'10000_hierarchical': 0.15771812080536912,
  'baseline_10k': 0.1610738255033557,
  'S2_atomic': 0.1610738255033557,
  'S3_window3': 0.1342281879194631,
  'S4_president': 0.18456375838926176,
  'S8_vote': 0.22818791946308725,
  'fixed_500': 0.2181208053691275,
  'fixed_1000': 0.174496644295302,
  'fixed_2000': 0.14429530201342283,
  'S9_topic': 0.21140939597315436,
  'S9_topic_noVotes': 0.3724832214765101,
  'S10_amendment': 0.22483221476510068,
  'S11_hybrid': 0.12080536912751678,
  'S11_hybrid_noVotes': 0.13758389261744966},
 1: {'10000_hierarchical': 0.0,
  'baseline_10k': 0.0,
  'S2_atomic': 0.0,
  'S3_window3': 0.0,
  'S4_president': 0.0,
  'S8_vote': 0.0,
  'fixed_500': 0.0,
  'fixed_1000': 0.0,
  'fixed_2000': 0.0,
  'S9_topic': 0.0,
  'S9_topic_noVotes': 0.14285714285714285,
  'S10_amendment': 0.14285714285714285,
  'S11_hybrid': 0.0,
  'S11_hybrid_noVotes': 0.0},
 2: {'10000_hierarchical': 0.65,
  'baseline_10k': 0.65,
  'S2_atomic': 0.42992353412597906,
  'S3_window3': 0.

In [19]:
# Per-routing: best strategy per cluster (each routing gets its own aligned map)
cluster_to_strategy_by_routing = {}
for rk in ROUTING_CONFIGS:
    cmap = {}
    for cid, scores in cluster_scores_by_routing[rk].items():
        valid = {s: v for s, v in scores.items() if not np.isnan(v)}
        cmap[cid] = max(valid, key=valid.get) if valid else BASELINE
    cluster_to_strategy_by_routing[rk] = cmap

# Backward-compat alias (used by UMAP visualization and per-cluster transfer analysis)
cluster_to_strategy = cluster_to_strategy_by_routing["no_rerouting"]

# ── How often does routing change the optimal strategy? ──────────────────────
print("Clusters where routing changes the optimal strategy vs no_rerouting:\n")
for rk in ["qre", "ums", "qre_ums"]:
    n_changed = sum(
        cluster_to_strategy_by_routing["no_rerouting"][cid] !=
        cluster_to_strategy_by_routing[rk][cid]
        for cid in cluster_train_qids
    )
    print(f"  no_rerouting → {rk}: {n_changed}/{len(cluster_train_qids)} clusters change")

# ── Summary table (no_rerouting map, backward compat) ────────────────────────
print()
summary_rows = []
for cid in sorted(cluster_to_strategy):
    n_q = len(cluster_train_qids[cid])
    best = cluster_to_strategy[cid]
    best_score = cluster_scores[cid].get(best, float("nan"))
    baseline_score = cluster_scores[cid].get(BASELINE, float("nan"))
    summary_rows.append({
        "cluster": cid,
        "n_train": n_q,
        "best_strategy": best,
        "coverage@3_best": round(best_score, 3),
        "coverage@3_baseline": round(baseline_score, 3),
        "delta": round(best_score - baseline_score, 3),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


Clusters where routing changes the optimal strategy vs no_rerouting:

  no_rerouting → qre: 4/21 clusters change
  no_rerouting → ums: 13/21 clusters change
  no_rerouting → qre_ums: 13/21 clusters change

 cluster  n_train      best_strategy  coverage@3_best  coverage@3_baseline  delta
      -1      123 10000_hierarchical            0.492                0.492  0.000
       0      149   S9_topic_noVotes            0.372                0.158  0.215
       1        7   S9_topic_noVotes            0.143                0.000  0.143
       2       10            S8_vote            0.950                0.650  0.300
       3       79   S9_topic_noVotes            0.297                0.101  0.196
       4       15       baseline_10k            0.600                0.567  0.033
       5       74 10000_hierarchical            0.527                0.527  0.000
       6        9           S9_topic            0.667                0.500  0.167
       7        7 10000_hierarchical            0.571   

In [20]:
# ── Re-score with per-routing score_k ────────────────────────────────────────
# ums / qre_ums force equal distribution across ≤3 collections.
# SCORE_K=3 → 1 slot/collection (binary hit/miss, noisy signal).
# Fix: score_k=9 → 3 slots/collection (same per-collection resolution as no_rerouting@3).

SCORE_K_PER_ROUTING = {"no_rerouting": 3, "qre": 3, "ums": 6, "qre_ums": 6}

cluster_scores_by_routing_v2 = {
    rk: {cid: {} for cid in cluster_train_qids}
    for rk in ROUTING_CONFIGS
}

print("Re-scoring with per-routing score_k:", SCORE_K_PER_ROUTING)
for sid in STRATEGY_IDS:
    strat_df = load_jsonl_as_df(sid)
    strat_uids = set(strat_df["unique_id"].tolist())

    for rk, rcfg in ROUTING_CONFIGS.items():
        rk_score_k = SCORE_K_PER_ROUTING[rk]
        routed_full = rcfg["fn"](strat_df.copy(), n_results=rk_score_k, **rcfg["kwargs"])

        for cid, qids in cluster_train_qids.items():
            valid_qids = qids & strat_uids
            subset = routed_full[routed_full["unique_id"].isin(valid_qids)]
            if len(subset) == 0:
                cluster_scores_by_routing_v2[rk][cid][sid] = float("nan")
                continue
            cov = compute_coverage_row_list(
                subset, k=rk_score_k, collections_dict=gold_collections,
                docs_col=rcfg["docs_col"], sources_col=rcfg["sources_col"],
            )
            cluster_scores_by_routing_v2[rk][cid][sid] = float(cov[2].mean())

    print(f"  {sid} done")

# Overwrite so all downstream cells pick up the corrected scores
cluster_scores_by_routing = cluster_scores_by_routing_v2
cluster_scores = cluster_scores_by_routing["no_rerouting"]

# Re-compute best strategy per cluster
cluster_to_strategy_by_routing = {}
for rk in ROUTING_CONFIGS:
    cmap = {}
    for cid, scores in cluster_scores_by_routing[rk].items():
        valid = {s: v for s, v in scores.items() if not np.isnan(v)}
        cmap[cid] = max(valid, key=valid.get) if valid else BASELINE
    cluster_to_strategy_by_routing[rk] = cmap
cluster_to_strategy = cluster_to_strategy_by_routing["no_rerouting"]

print("\nClusters where routing changes the optimal strategy (v2):")
for rk in ["qre", "ums", "qre_ums"]:
    n_ch = sum(
        cluster_to_strategy_by_routing["no_rerouting"][cid] !=
        cluster_to_strategy_by_routing[rk][cid]
        for cid in cluster_train_qids
    )
    print(f"  no_rerouting -> {rk}: {n_ch}/{len(cluster_train_qids)}")

print("\nFraction of clusters where best == baseline (v2):")
for rk in ROUTING_CONFIGS:
    n_same = sum(cluster_to_strategy_by_routing[rk][cid] == BASELINE for cid in cluster_train_qids)
    print(f"  {rk:<20s}: {n_same}/{len(cluster_train_qids)} ({100*n_same/len(cluster_train_qids):.0f}%)")
print("Done. cluster_scores_by_routing updated.")


Re-scoring with per-routing score_k: {'no_rerouting': 3, 'qre': 3, 'ums': 6, 'qre_ums': 6}


  10000_hierarchical done


  baseline_10k done


  S2_atomic done


  S3_window3 done


...

  S9_topic done


  S9_topic_noVotes done


  S10_amendment done


  S11_hybrid done


  S11_hybrid_noVotes done

Clusters where routing changes the optimal strategy (v2):
  no_rerouting -> qre: 4/21
  no_rerouting -> ums: 12/21
  no_rerouting -> qre_ums: 12/21

Fraction of clusters where best == baseline (v2):
  no_rerouting        : 8/21 (38%)
  qre                 : 8/21 (38%)
  ums                 : 13/21 (62%)
  qre_ums             : 14/21 (67%)
Done. cluster_scores_by_routing updated.


In [22]:
# ── Summary table (no_rerouting map, backward compat) ────────────────────────
print()
summary_rows = []
for cid in sorted(cluster_to_strategy):
    n_q = len(cluster_train_qids[cid])
    best = cluster_to_strategy[cid]
    best_score = cluster_scores[cid].get(best, float("nan"))
    baseline_score = cluster_scores[cid].get(BASELINE, float("nan"))
    summary_rows.append({
        "cluster": cid,
        "n_train": n_q,
        "best_strategy": best,
        "coverage@3_best": round(best_score, 3),
        "coverage@3_baseline": round(baseline_score, 3),
        "delta": round(best_score - baseline_score, 3),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


 cluster  n_train      best_strategy  coverage@3_best  coverage@3_baseline  delta
      -1      123 10000_hierarchical            0.492                0.492  0.000
       0      149   S9_topic_noVotes            0.372                0.158  0.215
       1        7   S9_topic_noVotes            0.143                0.000  0.143
       2       10            S8_vote            0.950                0.650  0.300
       3       79   S9_topic_noVotes            0.297                0.101  0.196
       4       15       baseline_10k            0.600                0.567  0.033
       5       74 10000_hierarchical            0.527                0.527  0.000
       6        9           S9_topic            0.667                0.500  0.167
       7        7 10000_hierarchical            0.571                0.571  0.000
       8       21            S8_vote            0.619                0.524  0.095
       9        6 10000_hierarchical            0.250                0.250  0.000
      10       

## 6. Assign Test Questions to Nearest Cluster

Project test embeddings into the training UMAP space, then find the nearest centroid.
All clusters — including the noise group (−1) — are in the centroid matrix, so every
test question is assigned to a real cluster-optimal strategy with no hardcoded fallback.


In [23]:
X_test = np.array(test_df["embeddings"].tolist())

# Transform test points using the already-fit reducer (no refit)
X_test_umap = reducer.transform(X_test)

# Centroid matrix: shape (n_clusters, CLUSTER_DIM)
cluster_ids_ordered = sorted(centroids.keys())
centroid_matrix = np.array([centroids[cid] for cid in cluster_ids_ordered])

# Nearest centroid assignment
dists = euclidean_distances(X_test_umap, centroid_matrix)
nearest_idx = dists.argmin(axis=1)
assigned_clusters = [cluster_ids_ordered[i] for i in nearest_idx]

test_df = test_df.copy()
test_df["assigned_cluster"] = assigned_clusters

# Per-routing optimal strategy assignment — each routing uses its own aligned cluster map
for rk in ROUTING_CONFIGS:
    col = f"assigned_strategy_{rk}"
    test_df[col] = [
        cluster_to_strategy_by_routing[rk].get(cid, BASELINE)
        for cid in assigned_clusters
    ]

# Backward-compat alias (no_rerouting map, used by per-cluster analysis)
test_df["assigned_strategy"] = test_df["assigned_strategy_no_rerouting"]

print("Strategy distribution by routing (test set):")
for rk, rcfg in ROUTING_CONFIGS.items():
    print(f"\n  [{rcfg['label']}]")
    print(test_df[f"assigned_strategy_{rk}"].value_counts().to_string())


Strategy distribution by routing (test set):

  [No Rerouting]
assigned_strategy_no_rerouting
10000_hierarchical    103
S9_topic_noVotes       70
S8_vote                36
baseline_10k            6
S9_topic                4

  [Query Rerouting (QRe)]
assigned_strategy_qre
10000_hierarchical    129
fixed_500              43
S8_vote                36
baseline_10k            6
S9_topic                4
S3_window3              1

  [UMS (Forced Equal)]
assigned_strategy_ums
10000_hierarchical    146
S9_topic               46
baseline_10k           22
S8_vote                 4
S3_window3              1

  [QRe + UMS]
assigned_strategy_qre_ums
10000_hierarchical    189
baseline_10k           22
S8_vote                 4
S9_topic                3
S3_window3              1


## 7. Build Evaluation Conditions (8 Conditions)

**8 conditions** = 4 routing strategies × 2 chunking policies.

| | **No Rerouting** | **QRe** | **UMS** | **QRe + UMS** |
|---|---|---|---|---|
| **Global Baseline** (uniform chunking) | ✓ | ✓ | ✓ | ✓ |
| **Task-Conditioned** (cluster-aligned) | ✓ | ✓ | ✓ | ✓ |

**Key fix vs. previous 4-condition version:** each TC condition uses the cluster→strategy map *trained under the same routing* applied at test time.  
This eliminates the mismatch that caused TC+QRe to underperform vs. the baseline.


In [29]:
test_df

,question,embeddings,unique_id,id_2,id_1,source_1,gold_ids,question_type,question_index,source_2,assigned_cluster,assigned_strategy_no_rerouting,assigned_strategy_qre,assigned_strategy_ums,assigned_strategy_qre_ums,assigned_strategy,baseline_strategy
0,Comment M. Blanc justifie-t-il l'importance de...,"[-0.0022237906232476234, 0.04156947508454323, ...",1_18870112_1887-01-11_004_000.txt_1,1887-01-11_004_000.txt,18870112,Le Gaulois,"[18870112, 1887-01-11_004_000.txt]",MH - Generic,1,Les Débats,8,S8_vote,S8_vote,baseline_10k,baseline_10k,S8_vote,10000_hierarchical
1,Quelle position M. Goblet adopte-t-il face à l...,"[0.006911382079124451, 0.022054297849535942, 0...",4_18870118_1887-01-17_016_002.txt_1,1887-01-17_016_002.txt,18870118,Le Gaulois,"[18870118, 1887-01-17_016_002.txt]",MH - Generic,1,Les Débats,16,10000_hierarchical,10000_hierarchical,10000_hierarchical,10000_hierarchical,10000_hierarchical,10000_hierarchical
2,Comment M. Achard justifie-t-il l'urgence de r...,"[0.04253409802913666, 0.07026002556085587, 0.0...",5_18870118_1887-01-17_016_002.txt_2,1887-01-17_016_002.txt,18870118,Le Gaulois,"[18870118, 1887-01-17_016_002.txt]",MH - Generic,2,Les Débats,16,10000_hierarchical,10000_hierarchical,10000_hierarchical,10000_hierarchical,10000_hierarchical,10000_hierarchical
3,"Selon le baron de Mackau, quelles sont les con...","[-0.06214575842022896, 0.00920034945011139, 0....",10_18870125_1887-01-24_002_003.txt_1,1887-01-24_002_003.txt,18870125,Le Gaulois,"[18870125, 1887-01-24_002_003.txt]",MH - Generic,1,Les Débats,5,10000_hierarchical,10000_hierarchical,10000_hierarchical,10000_hierarchical,10000_hierarchical,10000_hierarchical
4,Quelle est la perception de la presse concerna...,"[-0.0033271650318056345, 0.0105939581990242, -...",13_18870126_1887-01-25_006_003.txt_1,1887-01-25_006_003.txt,18870126,Le Gaulois,"[18870126, 1887-01-25_006_003.txt]",MH - Generic,1,Les Débats,12,S8_vote,S8_vote,10000_hierarchical,10000_hierarchical,S8_vote,10000_hierarchical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214,En quoi les deux journaux diffèrent-ils dans l...,"[-0.05277420952916145, 0.004449712112545967, -...",852_18870125_18870126_1,18870126,18870125,Le Gaulois,"[18870125, 18870126]",MH - Generic,1,L'Intransigeant,3,S9_topic_noVotes,10000_hierarchical,10000_hierarchical,10000_hierarchical,S9_topic_noVotes,10000_hierarchical
215,"En 1887, comment L'Intransigeant et Le Gaulois...","[-0.019408350810408592, 0.039293959736824036, ...",856_18870311_18870314_2,18870314,18870311,Le Gaulois,"[18870311, 18870314]",MH - Generic,2,L'Intransigeant,0,S9_topic_noVotes,fixed_500,S9_topic,10000_hierarchical,S9_topic_noVotes,10000_hierarchical
216,"En 1887, comment L’Intransigeant et Le Gaulois...","[-0.013041038066148758, 0.0025937978643924, -0...",862_18870130_18870131_0,18870131,18870130,Le Gaulois,"[18870130, 18870131]",MH - Generic,0,L'Intransigeant,0,S9_topic_noVotes,fixed_500,S9_topic,10000_hierarchical,S9_topic_noVotes,10000_hierarchical
217,"En 1887, comment L’Intransigeant et Le Gaulois...","[-0.008412779308855534, 0.028527285903692245, ...",870_18870511_18870512_0,18870512,18870511,Le Gaulois,"[18870511, 18870512]",MH - Generic,0,L'Intransigeant,0,S9_topic_noVotes,fixed_500,S9_topic,10000_hierarchical,S9_topic_noVotes,10000_hierarchical


In [27]:
test_df["assigned_strategy_no_rerouting"].value_counts()

assigned_strategy_no_rerouting
10000_hierarchical    103
S9_topic_noVotes       70
S8_vote                36
baseline_10k            6
S9_topic                4
Name: count, dtype: int64

In [24]:
def load_condition_raw(df, strategy_col):
    """Load raw JSONL data for a condition, grouped by strategy to avoid reloading."""
    chunks = []
    for sid in df[strategy_col].unique():
        strat_df = load_jsonl_as_df(sid)
        qids = df[df[strategy_col] == sid]["unique_id"].tolist()
        subset = strat_df[strat_df["unique_id"].isin(qids)]
        chunks.append(subset)
    return pd.concat(chunks, ignore_index=True)


test_df["baseline_strategy"] = BASELINE

print("Loading raw data for all conditions...")
raw_baseline = load_condition_raw(test_df, "baseline_strategy")

raw_tc = {}
for rk in ROUTING_CONFIGS:
    col = f"assigned_strategy_{rk}"
    #col = 'assigned_strategy'
    #col = "baseline_strategy"  # for backward compat with per-cluster eval cell
    raw_tc[rk] = load_condition_raw(test_df, col)
    print(f"  TC/{rk}: loaded")

# Backward-compat aliases (used by per-cluster eval cell)
raw_A = raw_baseline.copy()
raw_C = raw_tc["no_rerouting"].copy()

# CONDITION_CONFIGS: key → (raw_df, routing_fn, routing_kwargs, docs_col, sources_col, label)
CONDITION_CONFIGS = {}
for rk, rcfg in ROUTING_CONFIGS.items():
    CONDITION_CONFIGS[f"baseline_{rk}"] = (
        raw_baseline.copy(), rcfg["fn"], rcfg["kwargs"],
        rcfg["docs_col"], rcfg["sources_col"],
        f"Global Baseline + {rcfg['label']}",
    )
    CONDITION_CONFIGS[f"tc_{rk}"] = (
        raw_tc[rk].copy(), rcfg["fn"], rcfg["kwargs"],
        rcfg["docs_col"], rcfg["sources_col"],
        f"Task-Conditioned + {rcfg['label']}",
    )

print(f"\n{len(CONDITION_CONFIGS)} conditions:")
for key, (*_, lbl) in CONDITION_CONFIGS.items():
    print(f"  [{key}] {lbl}")


Loading raw data for all conditions...
  TC/no_rerouting: loaded
  TC/qre: loaded
  TC/ums: loaded
  TC/qre_ums: loaded

8 conditions:
  [baseline_no_rerouting] Global Baseline + No Rerouting
  [tc_no_rerouting] Task-Conditioned + No Rerouting
  [baseline_qre] Global Baseline + Query Rerouting (QRe)
  [tc_qre] Task-Conditioned + Query Rerouting (QRe)
  [baseline_ums] Global Baseline + UMS (Forced Equal)
  [tc_ums] Task-Conditioned + UMS (Forced Equal)
  [baseline_qre_ums] Global Baseline + QRe + UMS
  [tc_qre_ums] Task-Conditioned + QRe + UMS


## 8. Results

In [25]:
results_rows = []

for cond_key, (raw_df, routing_fn, routing_kwargs, docs_col, sources_col, display_label) in CONDITION_CONFIGS.items():
    row = {"condition": cond_key, "label": display_label}
    for k in K_VALUES:
        df_k = routing_fn(raw_df.copy(), n_results=k, **routing_kwargs)

        # Coverage@k
        cov = compute_coverage_row_list(
            df_k, k=k, collections_dict=gold_collections,
            docs_col=docs_col, sources_col=sources_col,
        )
        row[f"coverage@{k}"] = round(float(cov[2].mean()), 3)

        # Recall@k
        recall = compute_recall_metrics_dataframe(df_k, k_values=[k])
        row[f"recall@{k}"] = round(float(np.mean(recall[f"recall@{k}"])), 3)

    results_rows.append(row)
    print(f"  {display_label} done")

results_df = pd.DataFrame(results_rows)
results_table = results_df.set_index("condition")   # indexed by condition key

coverage_cols = [f"coverage@{k}" for k in K_VALUES]
recall_cols   = [f"recall@{k}"   for k in K_VALUES]

# ── Full results table ────────────────────────────────────────────────────────
display_table = results_df.set_index("label")[coverage_cols + recall_cols]
display(display_table.style.highlight_max(axis=0, color="#c6efce").format("{:.3f}"))

# ── Δ (Task-Conditioned − Global Baseline) per routing strategy ───────────────
print("\nΔ Coverage@k and Recall@k  (Task-Conditioned − Global Baseline):")
delta_rows = []
for rk, rcfg in ROUTING_CONFIGS.items():
    b = results_table.loc[f"baseline_{rk}", coverage_cols + recall_cols]
    t = results_table.loc[f"tc_{rk}",       coverage_cols + recall_cols]
    delta_rows.append({"Routing": rcfg["label"], **(t - b).round(3).to_dict()})
delta_df = pd.DataFrame(delta_rows).set_index("Routing")
display(delta_df.style.background_gradient(cmap="RdYlGn", axis=None).format("{:+.3f}"))


100%|██████████| 219/219 [00:00<00:00, 19840.43it/s]


  Global Baseline + No Rerouting done


100%|██████████| 219/219 [00:00<00:00, 27072.79it/s]


  Task-Conditioned + No Rerouting done


100%|██████████| 219/219 [00:00<00:00, 21953.41it/s]


...

100%|██████████| 219/219 [00:00<00:00, 26769.81it/s]


  Task-Conditioned + QRe + UMS done


,coverage@3,coverage@5,coverage@10,recall@3,recall@5,recall@10
label,,,,,,
Global Baseline + No Rerouting,0.390,0.457,0.571,0.390,0.457,0.573
Task-Conditioned + No Rerouting,0.465,0.538,0.638,0.466,0.537,0.639
Global Baseline + Query Rerouting (QRe),0.495,0.578,0.687,0.500,0.580,0.689
Task-Conditioned + Query Rerouting (QRe),0.511,0.581,0.677,0.511,0.575,0.674
Global Baseline + UMS (Forced Equal),0.534,0.598,0.715,0.534,0.600,0.717
Task-Conditioned + UMS (Forced Equal),0.537,0.600,0.712,0.537,0.603,0.712
Global Baseline + QRe + UMS,0.546,0.607,0.731,0.548,0.610,0.733
Task-Conditioned + QRe + UMS,0.548,0.610,0.728,0.550,0.612,0.728



Δ Coverage@k and Recall@k  (Task-Conditioned − Global Baseline):


,coverage@3,coverage@5,coverage@10,recall@3,recall@5,recall@10
Routing,,,,,,
No Rerouting,+0.075,+0.081,+0.067,+0.076,+0.080,+0.066
Query Rerouting (QRe),+0.016,+0.003,-0.010,+0.011,-0.005,-0.015
UMS (Forced Equal),+0.003,+0.002,-0.003,+0.003,+0.003,-0.005
QRe + UMS,+0.002,+0.003,-0.003,+0.002,+0.002,-0.005


In [28]:
# ── Compact summary: Coverage@k — Baseline vs TC per routing strategy ────────
print("Coverage@k — Global Baseline vs Task-Conditioned\n")
header = f"{'Routing':<22} {'Chunking':<20} {'C@3':>6} {'C@5':>6} {'C@10':>7}"
print(header)
print("-" * len(header))
for rk, rcfg in ROUTING_CONFIGS.items():
    b_row = results_table.loc[f"baseline_{rk}"]
    t_row = results_table.loc[f"tc_{rk}"]
    for cond, row in [("Global Baseline", b_row), ("Task-Conditioned", t_row)]:
        print(f"  {rcfg['label']:<20} {cond:<20} "
              f"{row['coverage@3']:>6.3f} {row['coverage@5']:>6.3f} {row['coverage@10']:>7.3f}")
    delta = t_row[coverage_cols] - b_row[coverage_cols]
    sign = lambda v: f"+{v:.3f}" if v >= 0 else f"{v:.3f}"
    print(f"  {'':20} {'Δ':<20} "
          f"{sign(delta['coverage@3']):>6} {sign(delta['coverage@5']):>6} {sign(delta['coverage@10']):>7}")
    print()


Coverage@k — Global Baseline vs Task-Conditioned

Routing                Chunking                C@3    C@5    C@10
-----------------------------------------------------------------
  No Rerouting         Global Baseline       0.390  0.457   0.571
  No Rerouting         Task-Conditioned      0.465  0.538   0.638
                       Δ                    +0.075 +0.081  +0.067

  Query Rerouting (QRe) Global Baseline       0.495  0.578   0.687
  Query Rerouting (QRe) Task-Conditioned      0.511  0.581   0.677
                       Δ                    +0.016 +0.003  -0.010

  UMS (Forced Equal)   Global Baseline       0.534  0.598   0.715
  UMS (Forced Equal)   Task-Conditioned      0.537  0.600   0.712
                       Δ                    +0.003 +0.002  -0.003

  QRe + UMS            Global Baseline       0.546  0.607   0.731
  QRe + UMS            Task-Conditioned      0.548  0.610   0.728
                       Δ                    +0.002 +0.003  -0.003



### Naive Retriever (No Rerouting) — Gain Breakdown

For the no_rerouting condition, compute per-question Coverage@3 under both the global baseline
and the task-conditioned assignment, then aggregate the delta by:
- **cluster** — where in the embedding space the gains concentrate
- **winning strategy** — which strategies actually deliver gain on the test set

In [32]:
RK = "no_rerouting"
K = 3
rcfg = ROUTING_CONFIGS[RK]

# Per-question Coverage@K — baseline (uniform 10000_hierarchical)
b_routed = rcfg["fn"](raw_baseline.copy(), n_results=K, **rcfg["kwargs"])
b_cov = compute_coverage_row_list(
    b_routed, k=K, collections_dict=gold_collections,
    docs_col=rcfg["docs_col"], sources_col=rcfg["sources_col"],
)
b_per_q = pd.Series(np.asarray(b_cov[2]), index=b_routed["unique_id"].values, name="cov_baseline")

# Per-question Coverage@K — task-conditioned (per-cluster optimal strategy)
t_routed = rcfg["fn"](raw_tc[RK].copy(), n_results=K, **rcfg["kwargs"])
t_cov = compute_coverage_row_list(
    t_routed, k=K, collections_dict=gold_collections,
    docs_col=rcfg["docs_col"], sources_col=rcfg["sources_col"],
)
t_per_q = pd.Series(np.asarray(t_cov[2]), index=t_routed["unique_id"].values, name="cov_tc")

# Join with test_df metadata (cluster + assigned strategy under this routing)
gain_df = (
    test_df[["unique_id", "assigned_cluster", f"assigned_strategy_{RK}"]]
    .rename(columns={f"assigned_strategy_{RK}": "assigned_strategy"})
    .set_index("unique_id")
    .join(b_per_q)
    .join(t_per_q)
    .dropna(subset=["cov_baseline", "cov_tc"])
)
gain_df["delta"] = gain_df["cov_tc"] - gain_df["cov_baseline"]

print(f"Test questions scored: {len(gain_df)}")
print(f"Overall Coverage@{K}  baseline={gain_df['cov_baseline'].mean():.3f}  "
      f"tc={gain_df['cov_tc'].mean():.3f}  Δ={gain_df['delta'].mean():+.3f}")

Test questions scored: 219
Overall Coverage@3  baseline=0.390  tc=0.465  Δ=+0.075


In [33]:
# ── Gain by cluster ──────────────────────────────────────────────────────────
by_cluster = (
    gain_df.groupby("assigned_cluster")
           .agg(n_test=("delta", "size"),
                strategy=("assigned_strategy", "first"),
                cov_baseline=("cov_baseline", "mean"),
                cov_tc=("cov_tc", "mean"),
                delta=("delta", "mean"))
           .round(3)
           .sort_values("delta", ascending=False)
)
print("Gain by cluster (Naive Retriever, Coverage@3):\n")
print(by_cluster.to_string())

n_pos = int((by_cluster["delta"] > 0).sum())
n_neg = int((by_cluster["delta"] < 0).sum())
n_zero = int((by_cluster["delta"] == 0).sum())
print(f"\n  Clusters with gain:      {n_pos}")
print(f"  Clusters with loss:      {n_neg}")
print(f"  Clusters unchanged:      {n_zero}")

Gain by cluster (Naive Retriever, Coverage@3):

                  n_test            strategy  cov_baseline  cov_tc  delta
assigned_cluster                                                         
 6                     3            S9_topic         0.500   0.833  0.333
 0                    43    S9_topic_noVotes         0.151   0.372  0.221
 3                    27    S9_topic_noVotes         0.167   0.296  0.130
 18                    8             S8_vote         0.188   0.312  0.125
 12                   16             S8_vote         0.406   0.489  0.083
 4                     6        baseline_10k         0.500   0.583  0.083
-1                     5  10000_hierarchical         0.700   0.700  0.000
 17                   10  10000_hierarchical         0.400   0.400  0.000
 16                   14  10000_hierarchical         0.464   0.464  0.000
 15                    1            S9_topic         0.500   0.500  0.000
 14                   20  10000_hierarchical         0.475   0.4

In [34]:
# ── Gain by winning strategy ─────────────────────────────────────────────────
by_strategy = (
    gain_df.groupby("assigned_strategy")
           .agg(n_test=("delta", "size"),
                n_clusters=("assigned_cluster", "nunique"),
                cov_baseline=("cov_baseline", "mean"),
                cov_tc=("cov_tc", "mean"),
                delta=("delta", "mean"))
           .round(3)
           .sort_values("delta", ascending=False)
)
print("Gain by winning strategy (Naive Retriever, Coverage@3):\n")
print(by_strategy.to_string())

Gain by winning strategy (Naive Retriever, Coverage@3):

                    n_test  n_clusters  cov_baseline  cov_tc  delta
assigned_strategy                                                  
S9_topic                 4           2         0.500   0.750  0.250
S9_topic_noVotes        70           2         0.157   0.343  0.186
baseline_10k             6           1         0.500   0.583  0.083
S8_vote                 36           7         0.444   0.495  0.051
10000_hierarchical     103           8         0.519   0.519  0.000
